In [0]:
import time
import json
import requests
import pandas as pd

## Data Source: CQC Syndication API

For this Home/Domiciliary Care Market Analysis project, we used the **CQC Syndication API** to extract Kent-based 
domiciliary/personal care provider data.

**Base URL:** `https://api.service.cqc.org.uk/public/v1`

### Getting Access

1. Register for a free account at [api-portal.service.cqc.org.uk/signup](https://api-portal.service.cqc.org.uk/signup)
2. Subscribe to the **Syndication** product
3. Collect your subscription key from the portal's Profile page

In [0]:

BASE_URL = "https://api.service.cqc.org.uk/public/v1"

# Getting CQC subscription key / API key from the secrets in Databricks
CQC_SUBSCRIPTION_KEY = dbutils.secrets.get(
    catalog="domiciliarycare", schema="security", key="api_key"
)

HEADERS = {
    "Ocp-Apim-Subscription-Key": CQC_SUBSCRIPTION_KEY,
    "User-Agent": "HomeSafeKentPipeline/1.0",  
    "Accept": "application/json",
}
print("Confirmed - Key loaded from Unity Catalog secret.")

### API Response Handling

The `cqc_get()` function below is used to send requests to the CQC Syndication API and 
handle its response — including checking for and reacting appropriately to different 
status codes, rather than assuming every request succeeds.

Per CQC's own API documentation, requests to this endpoint can return the following 
status codes: 
- **200 (OK)**
- **400 (Bad Request)**
- **404 (Not Found)**
- **500 (Internal Server Error)**

We additionally handle :
- **401 (Unauthorized)** 
- **502 (Bad Gateway)** <br>
which are not listed in CQC endpoint documentation but occur at the API gateway level general practice. 

We also handle:
- **429 (Too Many Requests)** <br>
explicitly with a wait-and-retry mechanism, since CQC enforces a **rate limit of 100 requests per 5 seconds**. Rather than allowing the pipeline to fail when this limit is hit, our code pauses (using the `Retry-After` value 
CQC provides, or a 5-second default) and automatically retries — up to 3 attempts per 
request , so a single burst of rate limiting doesn't crash the full extraction run across hundreds of records.

In [0]:
def cqc_get(path, params=None, max_retries=3):
    url = f"{BASE_URL}{path}"
    resp = None
    for attempt in range(1, max_retries + 1):
        resp = requests.get(url, headers=HEADERS, params=params, timeout=30)

        if resp.status_code == 200:
            return resp.json()

        if resp.status_code == 404:
            print(f"404 Not Found - no record exists at {url}")
            return None

        if resp.status_code == 400:
            raise RuntimeError("400 Bad Request - check your request parameters.")

        if resp.status_code == 401:
            raise RuntimeError("401 Unauthorized - key missing or invalid.")

        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 5))
            print(f"Rate limited (429). Waiting {wait}s before retry {attempt}/{max_retries}...")
            time.sleep(wait)
            continue

        if resp.status_code in (500, 502):
            raise RuntimeError(f"{resp.status_code} -- CQC server-side error, not a request problem.")

        resp.raise_for_status()

    raise RuntimeError(f"Gave up after {max_retries} attempts (last status {resp.status_code if resp else 'n/a'}).")

### Fetching Location Data via the CQC API

We now fetch location data through the CQC Syndication API, filtered by:

- **Local Authority** - `Kent`
- **Regulated Activity** - `Personal care`

We filter specifically on **Personal care** as Home Safe is only interested in the **home care / domiciliary care** 
market — services delivered to a person in their own home. 

In [0]:

def fetch_kent_data(local_authority="Kent", regulated_activity="Personal care", per_page=1000, polite_delay=0.3):
    """Page through /locations filtered to Kent + Personal care."""
    params = [
        ("localAuthority", local_authority),
        ("regulatedActivity", regulated_activity),
        ("perPage", per_page),
    ]
    all_locations = []
    page = 1
    while True:
        page_params = params + [("page", page)]
        data = cqc_get("/locations", params=page_params)
        all_locations.extend(data["locations"])
        print(f"Page {data['page']}/{data['totalPages']} -- {len(data['locations'])} rows "
              f"(running total {len(all_locations)})")
        if not data.get("nextPageUri"):
            break
        page += 1
        time.sleep(polite_delay)
    return all_locations

kent_locations_summary = fetch_kent_data()
print(f"\nTotal Kent personal-care locations: {len(kent_locations_summary)}")


In [0]:
def fetch_location_detail(location_id):
    return cqc_get(f"/locations/{location_id}")

detail_records = []
failed_ids = []
for i, loc in enumerate(kent_locations_summary, 1):
    try:
        detail_records.append(fetch_location_detail(loc["locationId"]))
    except Exception as e:
        print(f"FAILED on {loc['locationId']} ({loc.get('locationName')}): {e}")
        failed_ids.append(loc["locationId"])
    if i % 50 == 0:
        print(f"Processed {i}/{len(kent_locations_summary)}...")
    time.sleep(0.3)

print(f"\nDone. Fetched {len(detail_records)} records. Failed: {len(failed_ids)}")

In [0]:
def flatten_location_full(rec):
    """One row per location -- everything included, lists joined into strings."""
    ratings = (rec.get("currentRatings") or {}).get("overall") or {}
    overall_rating = ratings.get("rating")
    overall_rating_date = ratings.get("reportDate")
    rating_framework = "legacy_ratings"

    if not overall_rating:
        assessment = rec.get("assessment") or []
        if assessment:
            asg_ratings = assessment[0].get("ratings", {}).get("asgRatings", [])
            if asg_ratings:
                overall_rating = asg_ratings[0].get("rating")
                overall_rating_date = assessment[0].get("assessmentPlanPublishedDateTime")
                rating_framework = "single_assessment"
            else:
                rating_framework = "not_yet_rated"
        else:
            rating_framework = "not_yet_rated"

    # keyQuestionRatings -- ADDED: each sub-rating individually, not just overall
    key_question_ratings = {kq.get("name"): kq.get("rating") for kq in ratings.get("keyQuestionRatings", []) or []}

    # regulatedActivities + their contacts -- ADDED: registered manager names
    activity_names = []
    manager_names = []
    for activity in rec.get("regulatedActivities", []) or []:
        activity_names.append(activity.get("name", ""))
        for contact in activity.get("contacts", []) or []:
            full_name = f"{contact.get('personTitle','')} {contact.get('personGivenName','')} {contact.get('personFamilyName','')}".strip()
            if full_name:
                manager_names.append(full_name)

    return {
        "location_id": rec.get("locationId"),
        "provider_id": rec.get("providerId"),
        "name": rec.get("name"),
        "postal_code": rec.get("postalCode"),
        "postal_address_line1": rec.get("postalAddressLine1"),
        "postal_address_line2": rec.get("postalAddressLine2"),
        "town_city": rec.get("postalAddressTownCity"),
        "region": rec.get("region"),
        "local_authority": rec.get("localAuthority"),
        "constituency": rec.get("constituency"),
        "latitude": rec.get("onspdLatitude"),                      
        "longitude": rec.get("onspdLongitude"),                   
        "registration_status": rec.get("registrationStatus"),
        "registration_date": rec.get("registrationDate"),
        "deregistration_date": rec.get("deregistrationDate"),
        "dormancy": rec.get("dormancy"),
        "care_home": rec.get("careHome"),
        "number_of_beds": rec.get("numberOfBeds"),
        "main_phone_number": rec.get("mainPhoneNumber"),
        "website": rec.get("website"),
        "inspection_directorate": rec.get("inspectionDirectorate"),
        "last_inspection_date": (rec.get("lastInspection") or {}).get("date"),
        "last_report_date": (rec.get("lastReport") or {}).get("publicationDate"),

        "regulated_activities": "; ".join(activity_names),
        "registered_manager_names": "; ".join(manager_names),    
        "gac_service_types": "; ".join(s.get("name", "") for s in rec.get("gacServiceTypes", []) or []),
        "specialisms": "; ".join(s.get("name", "") for s in rec.get("specialisms", []) or []), 
        "inspection_categories": "; ".join(c.get("name", "") for c in rec.get("inspectionCategories", []) or []),  
        "inspection_areas": "; ".join(a.get("inspectionAreaName", "") for a in rec.get("inspectionAreas", []) or []),  
        "report_dates": "; ".join(r.get("reportDate", "") for r in rec.get("reports", []) or []),  
        "report_uris": "; ".join(r.get("reportUri", "") for r in rec.get("reports", []) or []),   

        "overall_rating": overall_rating,
        "overall_rating_date": overall_rating_date,
        "rating_framework": rating_framework,
        "rating_safe": key_question_ratings.get("Safe"),           
        "rating_well_led": key_question_ratings.get("Well-led"),   
        "rating_caring": key_question_ratings.get("Caring"),     
        "rating_responsive": key_question_ratings.get("Responsive"),  
        "rating_effective": key_question_ratings.get("Effective"),   
    }

In [0]:
kent_provider_ids = sorted({r.get("providerId") for r in detail_records if r.get("providerId")})

def fetch_provider_detail(provider_id):
    return cqc_get(f"/providers/{provider_id}")

provider_records = []
failed_provider_ids = []
for i, pid in enumerate(kent_provider_ids, 1):
    try:
        provider_records.append(fetch_provider_detail(pid))
    except Exception as e:
        print(f"FAILED on provider {pid}: {e}")
        failed_provider_ids.append(pid)
    if i % 20 == 0:
        print(f"Fetched {i}/{len(kent_provider_ids)}...")
    time.sleep(0.3)

print(f"Done. Fetched {len(provider_records)} providers.")

In [0]:
def flatten_provider_full(rec):
    """One row per provider -- everything included, lists joined into strings."""
    overall = (rec.get("currentRatings") or {}).get("overall") or {}
    key_question_ratings = {kq.get("name"): kq.get("rating") for kq in overall.get("keyQuestionRatings", []) or []}

    return {
        "provider_id": rec.get("providerId"),
        "name": rec.get("name"),
        "ownership_type": rec.get("ownershipType"),
        "type": rec.get("type"),
        "brand_name": rec.get("brandName"),
        "companies_house_number": rec.get("companiesHouseNumber"),
        "charity_number": rec.get("charityNumber"),
        "registration_status": rec.get("registrationStatus"),
        "registration_date": rec.get("registrationDate"),
        "postal_code": rec.get("postalCode"),
        "town_city": rec.get("postalAddressTownCity"),
        "region": rec.get("region"),
        "local_authority": rec.get("localAuthority"),
        "latitude": rec.get("onspdLatitude"),
        "longitude": rec.get("onspdLongitude"),
        "location_ids": "; ".join(rec.get("locationIds", []) or []),  
        "contacts": "; ".join(
            f"{c.get('personTitle','')} {c.get('personGivenName','')} {c.get('personFamilyName','')}".strip()
            for c in rec.get("contacts", []) or []
        ),
        "regulated_activities": "; ".join(a.get("name", "") for a in rec.get("regulatedActivities", []) or []),
        "inspection_categories": "; ".join(c.get("name", "") for c in rec.get("inspectionCategories", []) or []),
        "inspection_areas": "; ".join(a.get("inspectionAreaName", "") for a in rec.get("inspectionAreas", []) or []),
        "report_dates": "; ".join(r.get("reportDate", "") for r in rec.get("reports", []) or []),
        "overall_rating": overall.get("rating"),
        "overall_rating_date": overall.get("reportDate"),
        "rating_safe": key_question_ratings.get("Safe"),
        "rating_well_led": key_question_ratings.get("Well-led"),
        "rating_caring": key_question_ratings.get("Caring"),
        "rating_responsive": key_question_ratings.get("Responsive"),
        "rating_effective": key_question_ratings.get("Effective"),
    }

In [0]:
locations_df = pd.DataFrame([flatten_location_full(r) for r in detail_records])
providers_df = pd.DataFrame([flatten_provider_full(r) for r in provider_records])

print(f"locations_df: {len(locations_df)} rows, {len(locations_df.columns)} columns")
print(f"providers_df: {len(providers_df)} rows, {len(providers_df.columns)} columns")

spark.createDataFrame(locations_df.astype(str)).write.format("delta").mode("overwrite") \
    .saveAsTable("domiciliarycare.bronze.cqc_kent_locations")
spark.createDataFrame(providers_df.astype(str)).write.format("delta").mode("overwrite") \
    .saveAsTable("domiciliarycare.bronze.cqc_kent_providers")

print("Saved both tables.")